In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv("../.env")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "사용자가 입력한 요리의 레시피를 생각해 주세요."),
        ("human", "{dish}")
    ]
)
model = ChatOpenAI(model="gpt-6-luna")
output_parser = StrOutputParser()

In [2]:
prompt_value = prompt.invoke({"dish": "카레"})
ai_message = model.invoke(prompt_value)
output = output_parser.invoke(ai_message)

print(output)

## 기본 카레 레시피 (3~4인분)

### 재료
- 카레 가루 또는 고형 카레 1봉지
- 돼지고기 200g
- 감자 2개
- 양파 1개
- 당근 ½개
- 물 700~800ml
- 식용유 약간

### 만드는 법
1. 감자, 양파, 당근과 돼지고기를 먹기 좋은 크기로 썰어요.
2. 냄비에 식용유를 두르고 양파를 볶다가 돼지고기를 넣어 겉면이 익을 때까지 볶아요.
3. 감자와 당근을 넣고 2~3분 더 볶은 뒤 물을 붓고 끓여요.
4. 재료가 부드러워지면 불을 약하게 줄이고 카레를 넣어 잘 풀어요.
5. 눌어붙지 않도록 저으며 5분 정도 끓이면 완성입니다. 밥 위에 얹어 드세요.


### Runnable의 실행 방법-invoke, stream, batch

In [3]:
chain = prompt | model | output_parser

for chunk in chain.stream({"dish": "카레"}):
    print(chunk, end="", flush=True)

## 기본 카레 레시피 (3~4인분)

### 재료
- 카레 가루 또는 고형 카레 1봉지
- 돼지고기나 닭고기 200g
- 감자 2개
- 양파 1개
- 당근 ½개
- 물 700~800ml
- 식용유 약간

### 만드는 법
1. 감자와 당근은 한입 크기로 썰고, 양파와 고기도 먹기 좋게 썰어요.
2. 냄비에 식용유를 두르고 고기와 양파를 볶아요.
3. 고기 겉면이 익으면 감자와 당근을 넣고 2~3분 더 볶아요.
4. 물을 붓고 재료가 부드러워질 때까지 15~20분 끓여요.
5. 불을 약하게 줄이고 카레를 넣어 잘 풀어요. 걸쭉해질 때까지 저어가며 5분 정도 더 끓이면 완성입니다.
6. 따뜻한 밥 위에 카레를 얹어 먹어요.

카레가 너무 되직하면 물을 조금 더 넣고, 더 풍부한 맛을 원하면 마지막에 우유를 약간 넣어도 좋아요.

In [4]:
# 또한, batch 메서드를 사용하면 여러 입력을 한꺼번에 처리할 수 있다.
chain = prompt | model | output_parser

outputs = chain.batch([{"dish": "카레"}, {"dish": "우동"}])
print(outputs)

['### 기본 카레 레시피 (3~4인분)\n\n**재료**\n- 카레가루 1봉지(약 100g)\n- 돼지고기 200g\n- 감자 2개\n- 양파 1개\n- 당근 ½개\n- 물 700~800ml\n- 식용유 약간\n\n**만드는 법**\n1. 돼지고기와 감자, 양파, 당근을 한입 크기로 썰어요.\n2. 냄비에 식용유를 두르고 양파를 볶다가 돼지고기를 넣어 겉면이 익을 때까지 볶아요.\n3. 감자와 당근을 넣고 2분 정도 더 볶은 뒤 물을 부어요.\n4. 끓기 시작하면 중불로 낮추고, 재료가 부드러워질 때까지 15~20분 끓여요.\n5. 불을 약하게 줄이고 카레가루를 조금씩 넣어 잘 풀어요. 5분 정도 더 끓이면 완성입니다.\n\n밥 위에 얹어 먹으면 맛있어요. 카레가 너무 되직하면 물을 조금 더 넣어 농도를 맞추세요.', '## 기본 우동 레시피 (1인분)\n\n**재료**\n- 우동면 1인분\n- 물 400ml\n- 다시마 1장 또는 다시팩 1개\n- 간장 1큰술\n- 맛술 1큰술\n- 소금 약간\n- 대파, 어묵, 유부, 튀김가루 등 원하는 고명\n\n**만드는 법**\n1. 냄비에 물과 다시마를 넣고 끓입니다. 끓기 직전에 다시마를 건져내고, 다시팩을 썼다면 안내 시간에 맞춰 우려낸 뒤 건져요.\n2. 간장과 맛술을 넣고 맛을 봅니다. 싱거우면 소금을 조금 더 넣어 국물을 맞춰요.\n3. 우동면을 넣고 제품에 적힌 시간만큼 끓입니다. 냉동면은 면이 풀릴 때까지 데우면 돼요.\n4. 그릇에 담고 대파와 어묵, 유부 등 고명을 올리면 완성입니다.\n\n**간단 팁:** 다시 재료가 없다면 물 400ml에 쯔유 2~3큰술을 넣어 국물을 만들면 편해요.']


### LCEL의 ‘|’로 다양한 Runnable 연결하기

In [1]:
# 예를 들어, Zero-shot CoT로 단계적으로 생각하게 한 다음, 그 결과에서 결론만 추출하는 코드를 작성해 보겠다.
# 먼저 model(ChatOpenAI)과 output_parser(StrOutputParser)를 준비
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv("../.env")
model = ChatOpenAI(model="gpt-6-luna")

output_parser = StrOutputParser()

In [2]:
# 첫 번째로, Zero-shot CoT로 단계적으로 생각하게 하는 Chain을 만든다.
cot_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "사용자의 질문에 단계적으로 답변해주세요."),
        ("human", "{question}"),
    ]
)

cot_chain = cot_prompt | model | output_parser

In [3]:
# 두 번째로, 단계적으로 생각한 답변에서 결론을 추출하는 Chain을 만든다.
summarize_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "단계적으로 생각한 답변에서 결론만 추출하세요."),
        ("human", "{text}")
    ]
)

summarize_chain = summarize_prompt | model | output_parser

In [4]:
# 두 개의 Chain을 연결한 Chain을 만들고 실행해 본다.
cot_summarize_chain = cot_chain | summarize_chain
output = cot_summarize_chain.invoke({"question": "10 + 2 * 3"})
print(output)

c:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


16
